# 07 — W3 packing-format isolation checks (per maintainer's diagnosis on #49893)

**Context**: notebook 06 found a NEW `AssertionError` (`param_data.shape`
`[3072, 96]` vs. `loaded_weight.shape` `[3072, 103]`) loading a mixed-precision
(`config_groups`) checkpoint as a `draft_model` on the PR #49900 fix branch.
Maintainer `harjothkhara` diagnosed the real cause precisely (see
`docs/vllm-bug-report-draft.md` / the PR thread): vLLM currently expects
**dense** W3 packing (`ceil(1024*3/32)=96` int32 columns, values split across
int32 boundaries — introduced in `compressed-tensors` 0.17.0). Our checkpoint's
W3 tensor has 103 columns — **whole-values-per-int32** packing
(`floor(32/3)=10` values/word), an older layout. The two layouts coincide
exactly for 4-bit (32/4 divides evenly), which is why our W4 layers loaded
fine and only W3 tripped the assert. **This is a packing-format/version
mismatch, not a `draft_model`-path or merged-weight bug** — our earlier
hypothesis was wrong, his diagnosis supersedes it.

**Three action items from his reply**:
1. Report the exact `llmcompressor`/`compressed-tensors`/`torch` versions
   that produced our checkpoint (we didn't capture this at the time —
   rebuilding fresh here with versions captured immediately, and being
   explicit about pinning `compressed-tensors>=0.17.0` this time, per his
   suggested remediation).
2. **Isolation check 1**: a single-scheme **W3** checkpoint (uniform, not
   mixed) loaded as a **plain model** (`speculative_config=None`) — if this
   hits the identical 96-vs-103 assert, it proves the failure is independent
   of the draft-model path / this PR entirely.
3. **Isolation check 2**: a `config_groups` checkpoint with **both** groups at
   W4 (degenerate mixed-precision, packing unambiguous) loaded as a
   `draft_model` — should load end-to-end on the fix branch, confirming the
   PR's actual target-matching fix works cleanly once packing ambiguity isn't
   in play.

**Environment note**: sections 1-2 below (building checkpoints, isolation
check 1) do NOT need the fix branch's source-built vLLM — isolation check 1
is about plain quantized-checkpoint loading, unrelated to `draft_model` or
this PR, so a stock `pip install vllm` works fine and doesn't conflict with
`llmcompressor`'s `torch<=2.12.0` pin (this is the same simple setup notebook
06 sections 1-4 originally used, before the fix-branch/source-build saga).
Only section 3 (isolation check 2) needs the fix branch specifically, in a
**separate session**, handed off via Drive — same pattern as notebook 06's
"6-prep" section, for the same reason (fix branch's `torch==2.13.0` pin
conflicts with `llmcompressor`).

## 1. Build checkpoints + capture exact versions

One session, no fix-branch vLLM needed here. Builds both checkpoints needed
below: a uniform W3 single-scheme checkpoint (for isolation check 1) and a
W4/W4 `config_groups` checkpoint (for isolation check 2, degenerate
mixed-precision — structurally a `config_groups` checkpoint, but unambiguous
packing since both groups are 4-bit).

In [ ]:
# Explicit compressed-tensors>=0.17.0 pin this time, per the maintainer's
# suggested remediation -- and captured immediately so we can answer his
# version question precisely, unlike notebook 06 where we never checked.
!pip install -q llmcompressor "compressed-tensors>=0.17.0"

!pip show llmcompressor compressed-tensors torch | grep -E "^(Name|Version)"

import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

MODEL_ID = 'Qwen/Qwen3-0.6B'  # same tiny model as notebook 06, for continuity
SEQ_LEN = 2048
CALIB_N = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def get_calib_dataset():
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True).shuffle(seed=42, buffer_size=10_000)
    samples, collected = [], 0
    for item in ds:
        enc = tokenizer(item['text'], return_tensors='pt', truncation=True, max_length=SEQ_LEN)
        if enc['input_ids'].shape[1] == SEQ_LEN:
            samples.append(enc['input_ids'])
            collected += 1
            if collected >= CALIB_N:
                break
    return Dataset.from_dict({'input_ids': torch.cat(samples, dim=0).tolist()})

### 1a. Uniform W3 single-scheme checkpoint (for isolation check 1)

In [ ]:
W3_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w3-uniform-repro')
W3_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

model_w3 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
calib_dataset = get_calib_dataset()

# Uniform W3, single config_groups entry -- same quantization strategy
# (group, group_size=128) as notebook 06's mixed checkpoints, just one scheme
# instead of two, and 3-bit instead of 4-bit this time (notebook 06's
# single-scheme test was W4A16 -- deliberately different here, since W3 is
# specifically the bit-width the maintainer flagged as ambiguous).
recipe_w3 = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'w3_group': {
            'targets': ['Linear'],
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 3, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_w3, dataset=calib_dataset, recipe=recipe_w3, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model_w3.save_pretrained(str(W3_CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(W3_CHECKPOINT_DIR))

from safetensors import safe_open
shard = next(W3_CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"W3 checkpoint saved to {W3_CHECKPOINT_DIR}, contains weight_packed: {has_packed}")
assert has_packed

# Print the actual packed shape here directly (no vLLM needed for this) --
# this alone answers whether compressed-tensors>=0.17.0 produces the 96- or
# 103-column layout, before we even try loading it into vLLM.
for k in keys:
    if 'gate_proj.weight_packed' in k or 'up_proj.weight_packed' in k:
        with safe_open(str(shard), framework='pt') as f:
            t = f.get_tensor(k)
        print(f"{k}: shape {tuple(t.shape)}")
        break

del model_w3
torch.cuda.empty_cache()

### 1b. W4/W4 `config_groups` checkpoint (for isolation check 2)

Degenerate mixed-precision: two `config_groups` entries, both at 4-bit.
Structurally identical to notebook 06's real mixed checkpoint (two schemes,
layer-boundary split, same PR code path exercised), but packing is
unambiguous at 4-bit, so this isolates the target-matching fix from the
packing-format issue.

In [ ]:
import re

W4W4_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w4-config-groups-repro')
W4W4_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

model_w4w4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda', trust_remote_code=True)
calib_dataset = get_calib_dataset()

all_linear_names = [name for name, m in model_w4w4.named_modules()
                     if isinstance(m, torch.nn.Linear) and name != 'lm_head']

def layer_idx(name):
    m = re.match(r'model\.layers\.(\d+)\.', name)
    return int(m.group(1)) if m else -1

num_layers = max(layer_idx(n) for n in all_linear_names) + 1
mid_layer = num_layers // 2
group_a = [n for n in all_linear_names if layer_idx(n) < mid_layer]
group_b = [n for n in all_linear_names if layer_idx(n) >= mid_layer]
print(f"{len(group_a)} layers -> group_a (W4), {len(group_b)} layers -> group_b (W4)")

recipe_w4w4 = GPTQModifier(
    dampening_frac=0.01, ignore=['lm_head'],
    config_groups={
        'w4_group_a': {
            'targets': group_a,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 4, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
        'w4_group_b': {
            'targets': group_b,
            'input_activations': None, 'output_activations': None,
            'weights': {'num_bits': 4, 'type': 'int', 'symmetric': True, 'strategy': 'group', 'group_size': 128},
        },
    },
)
oneshot(model=model_w4w4, dataset=calib_dataset, recipe=recipe_w4w4, max_seq_length=SEQ_LEN, num_calibration_samples=CALIB_N)

model_w4w4.save_pretrained(str(W4W4_CHECKPOINT_DIR), save_compressed=True)
tokenizer.save_pretrained(str(W4W4_CHECKPOINT_DIR))

shard = next(W4W4_CHECKPOINT_DIR.glob('*.safetensors'))
with safe_open(str(shard), framework='pt') as f:
    keys = list(f.keys())
has_packed = any('weight_packed' in k for k in keys)
print(f"W4/W4 checkpoint saved to {W4W4_CHECKPOINT_DIR}, contains weight_packed: {has_packed}")
assert has_packed

del model_w4w4
torch.cuda.empty_cache()

### 1c. Copy both checkpoints to Drive (needed for section 3's separate session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/vllm-bug-repro-checkpoints
!cp -r checkpoints/qwen3-0.6b-w3-uniform-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/
!cp -r checkpoints/qwen3-0.6b-w4w4-config-groups-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/
!du -sh checkpoints/qwen3-0.6b-w3-uniform-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w3-uniform-repro
!du -sh checkpoints/qwen3-0.6b-w4w4-config-groups-repro /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4w4-config-groups-repro
# Verify the du -sh pairs match before moving on (Drive copies have been
# flaky mid-copy before -- see docs/context.md "Checkpoint storage").

## 2. Isolation check 1: uniform W3 checkpoint as a PLAIN model (no `speculative_config`)

Stays in the SAME session as section 1 -- stock `pip install vllm`, no
conflict with `llmcompressor` (this is the same simple setup notebook 06
sections 1-4 used before the fix-branch saga started). If this hits the
identical 96-vs-103 assert with no `draft_model`/`speculative_config`
involved at all, that confirms the maintainer's diagnosis: packing format,
not the PR.

In [ ]:
!pip install -q vllm
!pip show vllm | grep -E "^(Name|Version)"

# Known cu13/libcudart mismatch fix (docs/logs.md 2026-07-23) -- apply
# proactively in case this fresh environment hits it too.
import os, glob, subprocess
cu13_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/cu13/lib/libcudart.so.13')
if cu13_libs and not os.path.exists('/usr/lib/x86_64-linux-gnu/libcudart.so.13'):
    subprocess.run(['ln', '-sf', cu13_libs[0], '/usr/lib/x86_64-linux-gnu/libcudart.so.13'], check=True)
    subprocess.run(['ldconfig'], check=True)
    print("Applied libcudart cu13 symlink fix.")

In [ ]:
# Redefine if this is a fresh runtime / after the pip install above required a restart.
from pathlib import Path
W3_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w3-uniform-repro')

import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM

# Plain model load -- NO speculative_config, NO draft_model. Just loading a
# W3-quantized compressed-tensors checkpoint as an ordinary model.
llm_w3_plain = LLM(
    model=str(W3_CHECKPOINT_DIR.resolve()),
    max_model_len=2048,
)

print(f"ISOLATION CHECK 1: W3 checkpoint loaded as a plain model (no draft_model): {llm_w3_plain is not None}")
print("If this succeeded without the 96-vs-103 assert, that's a surprising result --")
print("worth double-checking the checkpoint's actual packed shape (cell 1a above) before reporting.")

## 3. Isolation check 2: W4/W4 `config_groups` checkpoint as a `draft_model`

**Needs a SEPARATE Colab session** with the PR #49900 fix branch's vLLM
installed (same reason as notebook 06's "6-prep": `llmcompressor` and this
build's `torch==2.13.0` pin are incompatible). Retry the precompiled install
first (cheap, fails fast if still 404); fall back to the full source build
(~1hr, real compute cost) if needed -- see notebook 06 section 6's install
cell for both commands.

In [ ]:
# In the SEPARATE fix-branch-vllm session:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf checkpoints
!mkdir -p checkpoints
!cp -r /content/drive/MyDrive/vllm-bug-repro-checkpoints/qwen3-0.6b-w4w4-config-groups-repro checkpoints/
!find checkpoints/qwen3-0.6b-w4w4-config-groups-repro
!du -sh checkpoints/qwen3-0.6b-w4w4-config-groups-repro
# Confirm model.safetensors shows up before proceeding.

In [ ]:
from pathlib import Path
MODEL_ID = 'Qwen/Qwen3-0.6B'
W4W4_CHECKPOINT_DIR = Path('checkpoints/qwen3-0.6b-w4w4-config-groups-repro')

import os
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

from vllm import LLM, SamplingParams

llm_w4w4_drafter = LLM(
    model=MODEL_ID,
    speculative_config={
        'method': 'draft_model',
        'model': str(W4W4_CHECKPOINT_DIR.resolve()),
        'num_speculative_tokens': 3,
    },
    max_model_len=2048,
)

print(f"ISOLATION CHECK 2: W4/W4 config_groups checkpoint loaded as draft_model: {llm_w4w4_drafter is not None}")

outputs = llm_w4w4_drafter.generate(
    ["The capital of France is"],
    SamplingParams(max_tokens=32, temperature=0),
)
for out in outputs:
    print("OUTPUT:", out.outputs[0].text)

## 4. Report back

Once sections 1-3 are done, reply on
[vllm-project/vllm#49893](https://github.com/vllm-project/vllm/issues/49893)
with:
- The captured `llmcompressor`/`compressed-tensors`/`torch` versions (cell 1,
  first cell's `pip show` output) and the packed tensor shape printed in 1a
  (confirms whether `compressed-tensors>=0.17.0` alone produces the 96-column
  dense layout).
- Isolation check 1's result (did the plain-model W3 load hit the same
  assert?).
- Isolation check 2's result (did the W4/W4 drafter load end-to-end and
  generate?).

Real numbers only, same rule as everywhere else in this project.